# 1D Euler's Elastica

In [9]:
import numpy as np
import dctkit as dt
import jax.numpy as jnp
from jax import jit, grad
from dctkit.math.opt import optctrl
from scipy import sparse
import os
from dctkit.dec import cochain as C
from dctkit.mesh import util

In [10]:
dt.config()

In [16]:
data = "data/xy_F_35.txt"
F = -35.
np.random.seed(42)

# load true data
data = np.genfromtxt(data)

# sampling factor for true data
sampling = 10

num_elements = 10

L = 1
h = L/(num_elements)

B = 7.854

A = F*L**2/B

In [37]:
def compute_true_solution(data, sampling, num_elements):
    x_true = data[:, 1][::sampling]
    y_true = data[:, 2][::sampling]
    theta_true = np.empty(num_elements, dtype=dt.float_dtype)
    for i in range(num_elements):
        theta_true[i] = np.arctan(
            (y_true[i+1]-y_true[i])/(x_true[i+1]-x_true[i]))

    return theta_true, x_true, y_true

In [17]:
# compute true solution
theta_true, x_true, y_true = compute_true_solution(data, sampling, num_elements)

In [20]:
num_nodes = num_elements + 1
mesh, _ = util.generate_line_mesh(num_nodes, 1.)
S = util.build_complex_from_mesh(mesh)
S.get_hodge_star()

# define internal cochain to compute the energy only in the interior points
int_vec = np.ones(S.num_nodes, dtype=dt.float_dtype)
int_vec[0] = 0
int_vec[-1] = 0
int_coch = C.CochainP0(complex=S, coeffs=int_vec)
# define the unit cochain on the dual nodes
ones_coch = C.CochainD0(complex=S, coeffs=np.ones(num_elements,dtype=dt.float_dtype))

# initial guess for the angles (EXCEPT PRESCRIBED ANGLE AT LEFT END)
theta_0 = np.zeros(num_elements-1, dtype=dt.float_dtype)

# cochain to zero residual on elements where BC is prescribed
mask = np.ones(num_elements, dtype=dt.float_dtype)
mask[0] = 0.

## Variational Formulation

## Non-variational formulation

In [38]:
def obj(x):
    # apply Dirichlet BC at left end
    theta = jnp.insert(x, 0, theta_true[0])
    theta_coch = C.CochainD0(S, theta)

    # dimensionless curvature at primal nodes (primal 0-cochain)
    dtheta = C.coboundary(theta_coch)
    print(C.star(dtheta).coeffs)
    curv = C.cochain_mul(int_coch, C.star(dtheta))

    load = C.scalar_mul(C.cos(theta_coch), A)

    # dimensionless bending moment
    moment = curv

    residual = C.sub(C.codifferential(C.star(moment)), load)

    return jnp.linalg.norm(residual.coeffs[1:])